In [ ]:
# using Pkg
# Pkg.add("LoggingExtras")

include("../Envs/Env.jl")
include("../Algorithms/PPO-RNN.jl")


## 1. Prepare Environment

In [2]:
using RockSample

pomdp = RockSamplePOMDP(7, 8)
pomdp_name = "RS78"
bool_full_observability = false
env = Env(pomdp, bool_full_observability)
action_space = GetActionSpace(env)
function create_env()
    return Env(pomdp, bool_full_observability)
end

# define convert_o function
function POMDPs.convert_o(T::Type{<:AbstractArray}, o::Int64, m::RockSamplePOMDP)
    vec = zeros(Float32, 3)
    vec[o] = 1.0f0
    return vec
end

# define process action function
function process_action(action::Int, action_space::UnitRange{Int})
    len = length(action_space)
    idx = action - first(action_space) + 1
    (idx < 1 || idx > len) && error("Action $action not in action space")
    onehot = zeros(Float32, len)
    onehot[idx] = 1.0f0
    return onehot
end

process_action (generic function with 1 method)

## 2. Prepare Parameters

In [3]:
state_dim = GetObsDim(env)
action_dim = length(action_space)
layer_size = 64
rnn_hidden_size = 64
gamma = discount(pomdp)
training_episodes = 100#10000
batch_size = 2048

RSState{8}([1, 1], Bool[0, 0, 1, 0, 0, 1, 0, 1])
Float32[1.0, 0.0, 0.0]


2048

## 3. Prepare PPO-RNN agent

In [4]:
# if want to use gpu, need to uncomment the below line, and use device=Flux.gpu
using CUDA

agent = PPORNNAgent(action_space, action_dim, state_dim;
    hidden_dim=layer_size, 
    rnn_hidden_size=rnn_hidden_size, 
    batch_size=batch_size, 
    device=Flux.cpu) 

PPORNNAgent(Chain(LSTM(16 => 64), Dense(64 => 64, tanh), Dense(64 => 13)), Chain(LSTM(16 => 64), Dense(64 => 64, tanh), Dense(64 => 1)), (layers = ((cell = (Wi = Leaf(Adam(eta=0.0001, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], (0.9, 0.999))), Wh = Leaf(Adam(eta=0.0001, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], (0.9, 0.999))), bias = Leaf(Adam(eta=0.0001, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], (0.9, 0.999)))),), (weight = Leaf(Ada

## 4. Train

In [5]:
# 训练
rewards, losses, evals = train!(create_env, agent, training_episodes)

Progress: 100%|█████████████████████████████████████████| Time: 0:11:45


Update 100 | Threads: 1 | Reward: 0.62 | Policy Loss: -0.0054 | Value Loss: 43.9449 | Eval: 7.35


(Float32[-1.5384616, -2.142857, -8.461538, -7.2727275, -2.5, -4.6153846, 2.3076923, -6.3636365, 0.8333333, -1.6666666  …  1.25, 6.428571, -0.71428573, 7.0588236, 4.117647, 7.0588236, -1.25, 8.571428, 11.666667, 0.625], Float32[-0.036110204, -0.017245702, -0.018082408, -0.013836115, -0.02257289, 0.00082445395, -0.021752901, -0.025905756, 0.017687397, -0.018570824  …  -0.0037716152, -0.007232035, -0.015174166, -0.013651401, -0.016161561, 0.0066539217, 0.011408402, -0.010226378, 0.017020801, -0.0053592804], Float32[43.052544, 37.90983, 40.3291, 37.469604, 36.32194, 38.162766, 60.7457, 45.631294, 40.96713, 36.84437  …  59.83347, 52.151886, 42.2898, 41.481686, 46.798077, 66.359726, 51.48801, 51.49508, 57.048805, 43.944885], Float32[7.350919])

## 5. Evaluation

In [6]:
evaluate(env, agent; num_episodes=10000, max_steps=100) 

7.350918906250964

## (Todo) Save or plot the data from Train (rewards, losses, evals)